# Datathon 2026 - Sales Forecasting (Full Option Pro)
## Mô hình: Detrended XGBoost

**Cải tiến:** Dự báo song song Revenue/COGS, đầy đủ đặc trưng Payday, Holiday, Sin/Cos và xử lý ngoại lệ 2021.

### 1. Import Libraries & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

TRAIN_PATH = '../Data/sales.csv'
SUBMISSION_SAMPLE_PATH = '../Data/sample_submission.csv'
OUTPUT_PATH = '../submission_v4_production_optimal.csv'
RANDOM_SEED = 42

### 2. Full Feature Engineering

In [ ]:
TET_DATES = pd.to_datetime([
    '2012-01-23', '2013-02-10', '2014-01-31', '2015-02-19', '2016-02-08',
    '2017-01-28', '2018-02-16', '2019-02-05', '2020-01-25', '2021-02-12',
    '2022-02-01', '2023-01-22', '2024-02-10'
])

def create_features(df):
    out = df.copy()
    out['Date'] = pd.to_datetime(out['Date'])
    out['month'] = out['Date'].dt.month
    out['day'] = out['Date'].dt.day
    out['dayofweek'] = out['Date'].dt.dayofweek
    out['dayofyear'] = out['Date'].dt.dayofyear
    
    # Sin/Cos encoding
    out['sin_year'] = np.sin(2 * np.pi * out['dayofyear'] / 365.25)
    out['cos_year'] = np.cos(2 * np.pi * out['dayofyear'] / 365.25)
    
    # Payday, Double Day, Weekend
    out['is_payday'] = ((out['day'] >= 25) | (out['day'] <= 5)).astype(int)
    out['is_double_day'] = (out['month'] == out['day']).astype(int)
    out['is_weekend'] = (out['dayofweek'] >= 5).astype(int)
    
    # Month Start/End
    out['is_month_start'] = out['Date'].dt.is_month_start.astype(int)
    out['is_month_end'] = out['Date'].dt.is_month_end.astype(int)

    # Holidays
    out['is_holiday'] = 0
    for m, d in [(1,1), (4,30), (5,1), (9,2), (12,25)]: 
        out.loc[(out['month']==m) & (out['day']==d), 'is_holiday'] = 1
        
    # Tết window
    out['days_to_tet'] = 999
    for tet in TET_DATES:
        diff = (out['Date'] - tet).dt.days
        mask = (diff >= -25) & (diff <= 30)
        out.loc[mask, 'days_to_tet'] = diff[mask]
    
    out['is_pre_tet'] = ((out['days_to_tet'] >= -15) & (out['days_to_tet'] < 0)).astype(int)
    out['is_post_tet'] = ((out['days_to_tet'] > 0) & (out['days_to_tet'] <= 20)).astype(int)
    
    return out

### 3. Modeling Logic

In [ ]:
XGB_PARAMS = {
    'n_estimators': 1334, 'learning_rate': 0.02, 'max_depth': 5,
    'subsample': 0.65, 'colsample_bytree': 0.73, 'min_child_weight': 10,
    'random_state': 42, 'objective': 'reg:squarederror', 'tree_method': 'hist'
}

def fit_predict_detrended(train_df, future_dates, target_col):
    tr = train_df.copy()
    tr['Date'] = pd.to_datetime(tr['Date'])
    tr['year'] = tr['Date'].dt.year
    annual_mean = tr.groupby('year')[target_col].mean()
    tr['norm_target'] = tr[target_col] / tr['year'].map(annual_mean)
    
    x_train = create_features(tr)
    x_future = create_features(pd.DataFrame({'Date': pd.to_datetime(future_dates)}))
    
    f_cols = ['month', 'day', 'dayofweek', 'is_weekend', 'sin_year', 'cos_year', 
              'is_payday', 'is_double_day', 'is_holiday', 'days_to_tet', 
              'is_pre_tet', 'is_post_tet', 'is_month_start', 'is_month_end']
    
    model = XGBRegressor(**XGB_PARAMS)
    model.fit(x_train[f_cols], tr['norm_target'])
    
    # Tính toán Growth Rate (Bỏ 2021)
    clean_means = annual_mean[annual_mean.index != 2021]
    if len(clean_means) > 1:
        growth = clean_means.pct_change().dropna().mean() + 1
    else:
        growth = 1.0
    if np.isnan(growth): growth = 1.0
    
    base_year = int(annual_mean.index.max())
    base_mean = float(annual_mean.loc[base_year])
    years_ahead = x_future['Date'].dt.year - base_year
    yearly_scale = base_mean * (growth ** years_ahead)
    
    preds_norm = model.predict(x_future[f_cols])
    return preds_norm * yearly_scale.values

### 4. Evaluation (Backtest Revenue & COGS)

In [ ]:
df_train = pd.read_csv(TRAIN_PATH, parse_dates=['Date']).sort_values('Date')
HORIZON = 548
split_date = df_train['Date'].max() - pd.Timedelta(days=HORIZON)
train_split = df_train[df_train['Date'] <= split_date]
valid_split = df_train[df_train['Date'] > split_date]

print("[*] Đang đánh giá mô hình...")
rev_val = fit_predict_detrended(train_split, valid_split['Date'], 'Revenue')
cogs_val = fit_predict_detrended(train_split, valid_split['Date'], 'COGS')

print(f"Revenue MAE: {mean_absolute_error(valid_split['Revenue'], rev_val):,.0f}")
print(f"COGS MAE:    {mean_absolute_error(valid_split['COGS'], cogs_val):,.0f}")

fig = go.Figure()
fig.add_trace(go.Scatter(x=valid_split['Date'], y=valid_split['Revenue'], name='Thực tế', opacity=0.3))
fig.add_trace(go.Scatter(x=valid_split['Date'], y=rev_val, name='Dự báo', line=dict(color='red')))
fig.update_layout(title='Backtest 2022 - Revenue', hovermode='x unified', xaxis=dict(rangeslider=dict(visible=True)))
fig.show()

### 5. Final Export

In [ ]:
df_sample = pd.read_csv(SUBMISSION_SAMPLE_PATH, parse_dates=['Date'])
print("[*] Đang tạo dự báo cuối cùng...")
rev_final = fit_predict_detrended(df_train, df_sample['Date'], 'Revenue')
cogs_final = fit_predict_detrended(df_train, df_sample['Date'], 'COGS')

submission = df_sample.copy()
submission['Revenue'] = rev_final
submission['COGS'] = cogs_final
submission['Date'] = submission['Date'].dt.strftime('%Y-%m-%d')
submission.to_csv(OUTPUT_PATH, index=False)
print(f"[+] Thành công! File submission đã lưu tại: {OUTPUT_PATH}")